In [1]:
import cv2
import numpy as np
import os
from pathlib import Path
import shutil
from tqdm import tqdm

def check_patch_quality(image_path, threshold=235, min_ratio=0.4):
    """
    检查 Patch 质量
    :param image_path: 图片路径
    :param threshold: 阈值（低于此值的像素被视为有内容/组织）
    :param min_ratio: 合格的最小占比（0.4 即 40%）
    """
    # 读取图片
    img = cv2.imread(str(image_path))
    if img is None:
        return False, 0
    
    # 转换为灰度图
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # 在病理图像中，组织通常比背景（白色 ~255）暗
    # 我们将像素值小于 threshold 的部分视为有效内容
    tissue_mask = (gray < threshold).astype(np.uint8)
    
    # 计算有效内容占比
    content_ratio = np.sum(tissue_mask) / (gray.shape[0] * gray.shape[1])
    
    return content_ratio >= min_ratio, content_ratio

def main():
    # 配置路径
    src_root = Path(r"G:\1.Sprouts\CRC.Pathology\20.WSL\Dataset\High\COAD.High.ROI.patches")
    dst_root = Path(r"G:\1.Sprouts\CRC.Pathology\20.WSL\Dataset\High\High.patches.qc")
    
    # 质控参数
    CONTENT_THRESHOLD = 230  # 判定为组织的灰度阈值（0-255，越小越严格）
    MIN_RATIO = 0.4          # 最小内容占比 40%
    
    # 如果目标根目录不存在则创建
    if not dst_root.exists():
        dst_root.mkdir(parents=True)
    
    # 遍历样本文件夹 (如 TCGA-A6-2678)
    sample_dirs = [d for d in src_root.iterdir() if d.is_dir()]
    
    print(f"🚀 开始质控处理，共发现 {len(sample_dirs)} 个样本。")
    
    for sample_dir in sample_dirs:
        sample_id = sample_dir.name
        save_dir = dst_root / sample_id
        save_dir.mkdir(parents=True, exist_ok=True)
        
        # 获取该样本下所有图片（支持 jpg, png）
        patch_files = list(sample_dir.glob("*.png")) + list(sample_dir.glob("*.jpg"))
        
        print(f"📦 正在处理样本: {sample_id} ({len(patch_files)} patches)")
        
        good_count = 0
        for patch_path in tqdm(patch_files, desc=f"Scanning {sample_id}", leave=False):
            is_qualified, ratio = check_patch_quality(patch_path, threshold=CONTENT_THRESHOLD, min_ratio=MIN_RATIO)
            
            if is_qualified:
                # 移动合格的 patch 到新目录
                shutil.move(str(patch_path), str(save_dir / patch_path.name))
                good_count += 1
        
        print(f"✅ 样本 {sample_id} 处理完成: 移至 QC 目录 {good_count} 张，过滤掉 {len(patch_files) - good_count} 张。")

if __name__ == "__main__":
    main()


🚀 开始质控处理，共发现 111 个样本。
📦 正在处理样本: TCGA-5M-AAT6 (27548 patches)


✅ 样本 TCGA-5M-AAT6 处理完成: 移至 QC 目录 16443 张，过滤掉 11105 张。
📦 正在处理样本: TCGA-5M-AATE (39204 patches)


✅ 样本 TCGA-5M-AATE 处理完成: 移至 QC 目录 27159 张，过滤掉 12045 张。
📦 正在处理样本: TCGA-A6-2676 (2420 patches)


✅ 样本 TCGA-A6-2676 处理完成: 移至 QC 目录 1781 张，过滤掉 639 张。
📦 正在处理样本: TCGA-A6-2681 (2350 patches)


✅ 样本 TCGA-A6-2681 处理完成: 移至 QC 目录 1470 张，过滤掉 880 张。
📦 正在处理样本: TCGA-A6-2683 (1927 patches)


✅ 样本 TCGA-A6-2683 处理完成: 移至 QC 目录 1339 张，过滤掉 588 张。
📦 正在处理样本: TCGA-A6-3808 (3660 patches)


... (102 outputs left out)

✅ 样本 TCGA-QG-A5YW 处理完成: 移至 QC 目录 0 张，过滤掉 0 张。
📦 正在处理样本: TCGA-QG-A5YX (0 patches)


✅ 样本 TCGA-QG-A5YX 处理完成: 移至 QC 目录 0 张，过滤掉 0 张。
📦 正在处理样本: TCGA-QL-A97D (0 patches)


✅ 样本 TCGA-QL-A97D 处理完成: 移至 QC 目录 0 张，过滤掉 0 张。
📦 正在处理样本: TCGA-WS-AB45 (0 patches)


✅ 样本 TCGA-WS-AB45 处理完成: 移至 QC 目录 0 张，过滤掉 0 张。
